# Home Credit - Future DPD Target Creation (30/60/90)

This notebook creates forward-looking targets using **future delinquency behavior** and past-only features.

Goal:
- Use **past months** (observation window) to create predictors
- Use **future months** (look-ahead window) to create labels

Key targets:
- `FUTURE_DPD30_FLAG`
- `FUTURE_DPD60_FLAG`
- `FUTURE_DEFAULT_90_FLAG`  (main default-like target)
- `TARGET_TRANSITION_TO_90` (customer not yet 90+ at obs, but reaches 90+ in future)


In [ ]:
from pathlib import Path
import gc
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)


In [ ]:
# pip install ipywidgets  # run in your environment if needed


In [ ]:
import kagglehub

kagglehub.login()

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('home-credit-default-risk')

print("Path to competition files:", path)

In [ ]:
# Paths
PROJECT_DIR = Path.cwd()
RAW_DIR = Path(path) if 'path' in globals() else Path('data/raw')
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "application_train": RAW_DIR / "application_train.csv",
    "application_test": RAW_DIR / "application_test.csv",
    "pos_cash": RAW_DIR / "POS_CASH_balance.csv",
    "credit_card": RAW_DIR / "credit_card_balance.csv",
    "bureau": RAW_DIR / "bureau.csv",
    "bureau_balance": RAW_DIR / "bureau_balance.csv",
    "installments_payments":RAW_DIR / "installments_payments.csv",
    "previous_application": RAW_DIR / "previous_application.csv",
    "credit_card":RAW_DIR / "credit_card_balance.csv"
}

missing = [k for k, v in FILES.items() if not v.exists()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

print("Project:", "PROJECT_ROOT")
for k, v in FILES.items():
    print(f"{k:20} -> {v.name}")


## 1) Modeling Timeline Parameters

Interpretation:
- `OBS_MONTH = -6` means observation is 6 months before reference month `0`
- Past features are computed only from months <= observation month
- Labels are computed from months > observation month up to look-ahead horizon


In [ ]:
# Timeline settings (editable)
OBS_MONTH = -6            # observation point
PAST_WINDOW = 12          # months used for past features: (OBS_MONTH-PAST_WINDOW, OBS_MONTH]
FUTURE_HORIZON = 6        # months used for target label: (OBS_MONTH, OBS_MONTH+FUTURE_HORIZON]

print(f"OBS_MONTH={OBS_MONTH}, PAST_WINDOW={PAST_WINDOW}, FUTURE_HORIZON={FUTURE_HORIZON}")


## 2) Build Customer-Month DPD Panel from POS + Credit Card

We combine delinquency signals:
- POS: `max(SK_DPD, SK_DPD_DEF)`
- Credit card: `max(SK_DPD, SK_DPD_DEF)`

Then `dpd_any = max(pos_dpd, cc_dpd)` per customer-month.


In [ ]:
# POS monthly DPD
pos = pd.read_csv(
    FILES["pos_cash"],
    usecols=["SK_ID_CURR", "MONTHS_BALANCE", "SK_DPD", "SK_DPD_DEF"],
    dtype={
        "SK_ID_CURR": "int32",
        "MONTHS_BALANCE": "int16",
        "SK_DPD": "float32",
        "SK_DPD_DEF": "float32",
    },
)

pos["pos_dpd"] = pos[["SK_DPD", "SK_DPD_DEF"]].max(axis=1)
pos_m = (
    pos.groupby(["SK_ID_CURR", "MONTHS_BALANCE"], as_index=False)["pos_dpd"]
       .max()
)

print("pos_m:", pos_m.shape)
pos_m.head()


In [ ]:
# Credit-card monthly DPD
cc = pd.read_csv(
    FILES["credit_card"],
    usecols=["SK_ID_CURR", "MONTHS_BALANCE", "SK_DPD", "SK_DPD_DEF"],
    dtype={
        "SK_ID_CURR": "int32",
        "MONTHS_BALANCE": "int16",
        "SK_DPD": "float32",
        "SK_DPD_DEF": "float32",
    },
)

cc["cc_dpd"] = cc[["SK_DPD", "SK_DPD_DEF"]].max(axis=1)
cc_m = (
    cc.groupby(["SK_ID_CURR", "MONTHS_BALANCE"], as_index=False)["cc_dpd"]
      .max()
)

print("cc_m:", cc_m.shape)
cc_m.head()


In [ ]:
# Merge monthly delinquency signals
panel = pos_m.merge(cc_m, on=["SK_ID_CURR", "MONTHS_BALANCE"], how="outer")
panel["dpd_any"] = panel[["pos_dpd", "cc_dpd"]].max(axis=1).fillna(0)

# Keep valid month order
panel = panel.sort_values(["SK_ID_CURR", "MONTHS_BALANCE"]).reset_index(drop=True)

print("panel:", panel.shape)
panel.head()


In [ ]:
# Optional quick sanity: DPD distribution
print(panel["dpd_any"].describe(percentiles=[0.5, 0.8, 0.9, 0.95, 0.99]))
print("Max observed DPD:", panel["dpd_any"].max())


## 3) Create Past-Only Features at Observation Point

In [ ]:
# Past window rows: (OBS_MONTH-PAST_WINDOW, OBS_MONTH]
past = panel[
    (panel["MONTHS_BALANCE"] <= OBS_MONTH)
    & (panel["MONTHS_BALANCE"] > (OBS_MONTH - PAST_WINDOW))
].copy()

past_feat = past.groupby("SK_ID_CURR").agg(
    past_obs_months=("dpd_any", "count"),
    past_dpd_max=("dpd_any", "max"),
    past_dpd_mean=("dpd_any", "mean"),
    past_dpd_std=("dpd_any", "std"),
    past_dpd30_count=("dpd_any", lambda x: (x >= 30).sum()),
    past_dpd60_count=("dpd_any", lambda x: (x >= 60).sum()),
    past_dpd90_count=("dpd_any", lambda x: (x >= 90).sum()),
).reset_index()

past_feat["past_dpd30_ratio"] = past_feat["past_dpd30_count"] / past_feat["past_obs_months"].replace(0, np.nan)
past_feat["past_dpd60_ratio"] = past_feat["past_dpd60_count"] / past_feat["past_obs_months"].replace(0, np.nan)
past_feat["past_dpd90_ratio"] = past_feat["past_dpd90_count"] / past_feat["past_obs_months"].replace(0, np.nan)

print("past_feat:", past_feat.shape)
past_feat.head()


In [ ]:
# Observation-stage state (latest known month <= OBS_MONTH)
obs_state = (
    panel[panel["MONTHS_BALANCE"] <= OBS_MONTH]
    .sort_values(["SK_ID_CURR", "MONTHS_BALANCE"], ascending=[True, False])
    .drop_duplicates("SK_ID_CURR")
    [["SK_ID_CURR", "MONTHS_BALANCE", "dpd_any"]]
    .rename(columns={"MONTHS_BALANCE": "obs_reference_month", "dpd_any": "dpd_at_obs"})
)

print("obs_state:", obs_state.shape)
obs_state.head()


## 4) Create Future Labels from Forward Window

In [ ]:
# Future window rows: (OBS_MONTH, OBS_MONTH+FUTURE_HORIZON]
future = panel[
    (panel["MONTHS_BALANCE"] > OBS_MONTH)
    & (panel["MONTHS_BALANCE"] <= (OBS_MONTH + FUTURE_HORIZON))
].copy()

future_label = future.groupby("SK_ID_CURR").agg(
    future_obs_months=("dpd_any", "count"),
    future_dpd_max=("dpd_any", "max"),
).reset_index()

future_label["FUTURE_DPD30_FLAG"] = (future_label["future_dpd_max"] >= 30).astype("int8")
future_label["FUTURE_DPD60_FLAG"] = (future_label["future_dpd_max"] >= 60).astype("int8")
future_label["FUTURE_DEFAULT_90_FLAG"] = (future_label["future_dpd_max"] >= 90).astype("int8")

print("future_label:", future_label.shape)
future_label.head()


In [ ]:
def dpd_stage(x: pd.Series) -> pd.Series:
    return np.select(
        [x >= 90, x >= 60, x >= 30],
        [3, 2, 1],
        default=0,
    ).astype("int8")

obs_state["OBS_STAGE"] = dpd_stage(obs_state["dpd_at_obs"])
future_label["FUTURE_STAGE_MAX"] = dpd_stage(future_label["future_dpd_max"])


## 4B) Add Additional Risk Features (Application, Bureau, Installments, Previous Apps, Cards, Excel Market Regime)

This block adds customer-level features from multiple Home Credit tables and merges sector-level market signals from `combined_output.xlsx`.


In [ ]:
FILES

In [ ]:
import os 
os.listdir(RAW_DIR)

In [ ]:
# Additional feature engineering from other tables + Excel market regime file
excel_file = Path("combined_output.xlsx")

# 1) Application-level static features (train + test)
app_cols = [
    "SK_ID_CURR", "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
    "DAYS_BIRTH", "DAYS_EMPLOYED", "DAYS_REGISTRATION", "DAYS_ID_PUBLISH",
    "CNT_CHILDREN", "CNT_FAM_MEMBERS", "REGION_POPULATION_RELATIVE",
    "REGION_RATING_CLIENT", "REGION_RATING_CLIENT_W_CITY",
    "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "ORGANIZATION_TYPE",
]

app_train = pd.read_csv(FILES["application_train"], usecols=app_cols, low_memory=False)
app_test = pd.read_csv(FILES["application_test"], usecols=app_cols, low_memory=False)
app_all = pd.concat([app_train, app_test], ignore_index=True)

# Clean and derive
app_all["DAYS_EMPLOYED"] = app_all["DAYS_EMPLOYED"].replace(365243, np.nan)
app_all["age_years"] = (-app_all["DAYS_BIRTH"] / 365.25).clip(lower=18, upper=100)
app_all["employment_years"] = (-app_all["DAYS_EMPLOYED"] / 365.25).clip(lower=0)
app_all["credit_income_ratio"] = app_all["AMT_CREDIT"] / app_all["AMT_INCOME_TOTAL"].replace(0, np.nan)
app_all["annuity_income_ratio"] = app_all["AMT_ANNUITY"] / app_all["AMT_INCOME_TOTAL"].replace(0, np.nan)
app_all["annuity_credit_ratio"] = app_all["AMT_ANNUITY"] / app_all["AMT_CREDIT"].replace(0, np.nan)
app_all["goods_credit_ratio"] = app_all["AMT_GOODS_PRICE"] / app_all["AMT_CREDIT"].replace(0, np.nan)
app_all["ext_score_mean"] = app_all[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].mean(axis=1)
app_all["ext_score_min"] = app_all[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].min(axis=1)

# Organization -> broad sector mapping (for merging market-sector stats)
org_to_sector = {
    "Agriculture": "Basic Materials",
    "Construction": "Industrials",
    "Industry: type 1": "Industrials", "Industry: type 2": "Industrials", "Industry: type 3": "Industrials",
    "Industry: type 4": "Industrials", "Industry: type 5": "Industrials", "Industry: type 6": "Industrials",
    "Industry: type 7": "Industrials", "Industry: type 8": "Industrials", "Industry: type 9": "Industrials",
    "Industry: type 10": "Industrials", "Industry: type 11": "Industrials", "Industry: type 12": "Industrials",
    "Industry: type 13": "Industrials",
    "Transport: type 1": "Industrials", "Transport: type 2": "Industrials", "Transport: type 3": "Industrials", "Transport: type 4": "Industrials",
    "Business Entity Type 1": "Financial Services", "Business Entity Type 2": "Financial Services", "Business Entity Type 3": "Financial Services",
    "Bank": "Financial Services", "Insurance": "Financial Services", "Realtor": "Real Estate",
    "Medicine": "Healthcare", "Hospital": "Healthcare",
    "Telecom": "Communication Services", "Advertising": "Communication Services",
    "Electricity": "Utilities",
    "Trade: type 1": "Consumer Cyclical", "Trade: type 2": "Consumer Cyclical", "Trade: type 3": "Consumer Cyclical",
    "Trade: type 4": "Consumer Cyclical", "Trade: type 5": "Consumer Cyclical", "Trade: type 6": "Consumer Cyclical", "Trade: type 7": "Consumer Cyclical",
    "Restaurant": "Consumer Cyclical", "Hotel": "Consumer Cyclical", "Services": "Consumer Defensive",
    "Government": "Utilities", "School": "Consumer Defensive", "Kindergarten": "Consumer Defensive",
    "University": "Consumer Defensive", "Military": "Utilities", "Police": "Utilities",
    "Postal": "Communication Services", "Security": "Industrials", "Security Ministries": "Utilities",
    "Housing": "Real Estate", "Culture": "Consumer Cyclical", "Mobile": "Technology", "Legal Services": "Financial Services",
    "Cleaning": "Industrials", "Emergency": "Utilities", "Self-employed": "Financial Services",
    "Other": "Financial Services", "XNA": "Financial Services",
}
app_all["org_sector"] = app_all["ORGANIZATION_TYPE"].map(org_to_sector).fillna("Financial Services")

# 2) Market regime features from Excel (sector daily averageChange)
if excel_file.exists():
    mkt = pd.read_excel(excel_file)
    mkt["date"] = pd.to_datetime(mkt["date"], errors="coerce")
    mkt = mkt[mkt["date"].notna()].copy()

    sector_stats = (
        mkt.groupby("sector")["averageChange"]
        .agg(
            sector_avg_change="mean",
            sector_volatility="std",
            sector_downside_20p=lambda x: np.nanpercentile(x, 20),
            sector_upside_80p=lambda x: np.nanpercentile(x, 80),
            sector_neg_day_ratio=lambda x: (x < 0).mean(),
        )
        .reset_index()
    )

    app_all = app_all.merge(sector_stats, left_on="org_sector", right_on="sector", how="left")
    app_all.drop(columns=["sector"], inplace=True, errors="ignore")

    # portfolio-wide regime constants (same for all customers)
    mkt = mkt.sort_values("date")
    latest_date = mkt["date"].max()
    last_90 = mkt[mkt["date"] >= (latest_date - pd.Timedelta(days=90))]
    market_regime = {
        "mkt_last90_mean_change": float(last_90["averageChange"].mean()),
        "mkt_last90_volatility": float(last_90["averageChange"].std()),
        "mkt_full_neg_day_ratio": float((mkt["averageChange"] < 0).mean()),
    }
else:
    sector_avg_change= np.nan,
    sector_volatility=np.nan,
    sector_downside_20p=np.nan,
    sector_upside_80p=np.nan,
    sector_neg_day_ratio=np.nan,

    market_regime = {
        "mkt_last90_mean_change": np.nan,
        "mkt_last90_volatility": np.nan,
        "mkt_full_neg_day_ratio": np.nan,
        
    }

app_feat_cols = [
    "SK_ID_CURR", "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
    "DAYS_BIRTH", "DAYS_EMPLOYED", "DAYS_REGISTRATION", "DAYS_ID_PUBLISH",
    "CNT_CHILDREN", "CNT_FAM_MEMBERS", "REGION_POPULATION_RELATIVE",
    "REGION_RATING_CLIENT", "REGION_RATING_CLIENT_W_CITY",
    "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
    "age_years", "employment_years", "credit_income_ratio", "annuity_income_ratio",
    "annuity_credit_ratio", "goods_credit_ratio", "ext_score_mean", "ext_score_min",
    "sector_avg_change", "sector_volatility", "sector_downside_20p", "sector_upside_80p", "sector_neg_day_ratio",
]
if excel_file.exists():
    print("combined_output exists, merging market regime features")
    app_feat = app_all[app_feat_cols].copy()
else:
    app_all[["sector_avg_change", "sector_volatility", "sector_downside_20p", "sector_upside_80p", "sector_neg_day_ratio"]] = np.nan
    app_feat = app_all[app_feat_cols].copy()
    

# 3) Bureau + bureau balance delinquency features
bureau = pd.read_csv(
    FILES["bureau"],
    usecols=[
        "SK_ID_CURR", "SK_ID_BUREAU", "CREDIT_ACTIVE", "DAYS_CREDIT", "DAYS_CREDIT_ENDDATE",
        "CREDIT_DAY_OVERDUE", "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM_OVERDUE",
    ],
    low_memory=False,
)

bureau["is_active"] = (bureau["CREDIT_ACTIVE"] == "Active").astype("int8")
bureau_feat = bureau.groupby("SK_ID_CURR").agg(
    buro_loan_count=("SK_ID_BUREAU", "count"),
    buro_active_loan_count=("is_active", "sum"),
    buro_days_credit_mean=("DAYS_CREDIT", "mean"),
    buro_days_enddate_mean=("DAYS_CREDIT_ENDDATE", "mean"),
    buro_day_overdue_max=("CREDIT_DAY_OVERDUE", "max"),
    buro_day_overdue_mean=("CREDIT_DAY_OVERDUE", "mean"),
    buro_credit_sum_total=("AMT_CREDIT_SUM", "sum"),
    buro_debt_sum_total=("AMT_CREDIT_SUM_DEBT", "sum"),
    buro_overdue_sum_total=("AMT_CREDIT_SUM_OVERDUE", "sum"),
).reset_index()
buro_active_denom = bureau_feat["buro_loan_count"].replace(0, np.nan)
buro_credit_denom = bureau_feat["buro_credit_sum_total"].replace(0, np.nan)
buro_debt_denom = bureau_feat["buro_debt_sum_total"].replace(0, np.nan)
bureau_feat["buro_active_ratio"] = bureau_feat["buro_active_loan_count"] / buro_active_denom
bureau_feat["buro_overdue_credit_ratio"] = bureau_feat["buro_overdue_sum_total"] / buro_credit_denom
bureau_feat["buro_debt_credit_ratio"] = bureau_feat["buro_debt_sum_total"] / buro_credit_denom

bb = pd.read_csv(FILES["bureau_balance"], usecols=["SK_ID_BUREAU", "STATUS"], low_memory=False)
status_map = {"X": 0, "C": 0, "0": 0, "1": 1, "2": 2, "3": 3, "4": 4, "5": 5}
bb["status_num"] = bb["STATUS"].map(status_map).fillna(0)
bb_buro = bb.groupby("SK_ID_BUREAU", as_index=False).agg(
    bb_status_max=("status_num", "max"),
    bb_status_mean=("status_num", "mean"),
)

bb_curr = bureau[["SK_ID_CURR", "SK_ID_BUREAU"]].merge(bb_buro, on="SK_ID_BUREAU", how="left")
bb_feat = bb_curr.groupby("SK_ID_CURR", as_index=False).agg(
    bb_status_max=("bb_status_max", "max"),
    bb_status_mean=("bb_status_mean", "mean"),
    bb_severe_count=("bb_status_max", lambda x: (x >= 3).sum()),
)

# 4) Installment payment behavior
inst = pd.read_csv(
    FILES["installments_payments"],
    usecols=["SK_ID_CURR", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT", "AMT_INSTALMENT", "AMT_PAYMENT"],
    low_memory=False,
)
inst["pay_delay_days"] = (inst["DAYS_ENTRY_PAYMENT"] - inst["DAYS_INSTALMENT"]).fillna(0)
inst["paid_ratio"] = inst["AMT_PAYMENT"] / inst["AMT_INSTALMENT"].replace(0, np.nan)
inst_feat = inst.groupby("SK_ID_CURR", as_index=False).agg(
    inst_count=("pay_delay_days", "count"),
    inst_delay_mean=("pay_delay_days", "mean"),
    inst_delay_max=("pay_delay_days", "max"),
    inst_late_count=("pay_delay_days", lambda x: (x > 0).sum()),
    inst_late30_count=("pay_delay_days", lambda x: (x > 30).sum()),
    inst_paid_ratio_mean=("paid_ratio", "mean"),
    inst_paid_ratio_min=("paid_ratio", "min"),
)
inst_denom = inst_feat["inst_count"].replace(0, np.nan)
inst_feat["inst_late_ratio"] = inst_feat["inst_late_count"] / inst_denom
inst_feat["inst_late30_ratio"] = inst_feat["inst_late30_count"] / inst_denom

# 5) Previous application behavior
prev = pd.read_csv(
    FILES["previous_application"],
    usecols=["SK_ID_CURR", "NAME_CONTRACT_STATUS", "AMT_APPLICATION", "AMT_CREDIT", "RATE_DOWN_PAYMENT", "DAYS_DECISION"],
    low_memory=False,
)
prev["approved"] = (prev["NAME_CONTRACT_STATUS"] == "Approved").astype("int8")
prev["refused"] = (prev["NAME_CONTRACT_STATUS"] == "Refused").astype("int8")
prev["prev_app_credit_ratio"] = prev["AMT_APPLICATION"] / prev["AMT_CREDIT"].replace(0, np.nan)

prev_feat = prev.groupby("SK_ID_CURR", as_index=False).agg(
    prev_count=("NAME_CONTRACT_STATUS", "count"),
    prev_approved_count=("approved", "sum"),
    prev_refused_count=("refused", "sum"),
    prev_days_decision_mean=("DAYS_DECISION", "mean"),
    prev_down_payment_mean=("RATE_DOWN_PAYMENT", "mean"),
    prev_app_credit_ratio_mean=("prev_app_credit_ratio", "mean"),
)
prev_denom = prev_feat["prev_count"].replace(0, np.nan)
prev_feat["prev_approved_ratio"] = prev_feat["prev_approved_count"] / prev_denom
prev_feat["prev_refused_ratio"] = prev_feat["prev_refused_count"] / prev_denom

# 6) Credit card behavior
cc_full = pd.read_csv(
    FILES["credit_card"],
    usecols=["SK_ID_CURR", "AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL", "AMT_PAYMENT_CURRENT", "AMT_INST_MIN_REGULARITY", "SK_DPD", "SK_DPD_DEF"],
    low_memory=False,
)
cc_full["cc_dpd_any"] = cc_full[["SK_DPD", "SK_DPD_DEF"]].max(axis=1)
cc_full["cc_utilization"] = cc_full["AMT_BALANCE"] / cc_full["AMT_CREDIT_LIMIT_ACTUAL"].replace(0, np.nan)
cc_full["cc_payment_ratio"] = cc_full["AMT_PAYMENT_CURRENT"] / cc_full["AMT_INST_MIN_REGULARITY"].replace(0, np.nan)

cc_feat = cc_full.groupby("SK_ID_CURR", as_index=False).agg(
    cc_dpd_max=("cc_dpd_any", "max"),
    cc_dpd_mean=("cc_dpd_any", "mean"),
    cc_util_mean=("cc_utilization", "mean"),
    cc_util_max=("cc_utilization", "max"),
    cc_payment_ratio_mean=("cc_payment_ratio", "mean"),
)

# Free memory
for _obj in [app_train, app_test, app_all, bureau, bb, bb_buro, bb_curr, inst, prev, cc_full]:
    del _obj
gc.collect()

print("app_feat:", app_feat.shape)
print("bureau_feat:", bureau_feat.shape, "| bb_feat:", bb_feat.shape)
print("inst_feat:", inst_feat.shape, "| prev_feat:", prev_feat.shape, "| cc_feat:", cc_feat.shape)
print("market_regime:", market_regime)


In [ ]:
!pip install openpyxl 

## 5) Build Final Training Table (Train/Test IDs + features + future target)

This output is what you can use for modeling where labels come from future behavior.


In [ ]:
train_ids = pd.read_csv(
    FILES["application_train"],
    usecols=["SK_ID_CURR", "TARGET"],
    dtype={"SK_ID_CURR": "int32", "TARGET": "float32"},
)
train_ids["is_train"] = 1

test_ids = pd.read_csv(
    FILES["application_test"],
    usecols=["SK_ID_CURR"],
    dtype={"SK_ID_CURR": "int32"},
)
test_ids["TARGET"] = np.nan
test_ids["is_train"] = 0

base = pd.concat([train_ids, test_ids], ignore_index=True)

# Core panel features
final_df = base.merge(obs_state, on="SK_ID_CURR", how="left")
final_df = final_df.merge(past_feat, on="SK_ID_CURR", how="left")
final_df = final_df.merge(future_label, on="SK_ID_CURR", how="left")

# Additional cross-table borrower features
final_df = final_df.merge(app_feat, on="SK_ID_CURR", how="left")
final_df = final_df.merge(bureau_feat, on="SK_ID_CURR", how="left")
final_df = final_df.merge(bb_feat, on="SK_ID_CURR", how="left")
final_df = final_df.merge(inst_feat, on="SK_ID_CURR", how="left")
final_df = final_df.merge(prev_feat, on="SK_ID_CURR", how="left")
final_df = final_df.merge(cc_feat, on="SK_ID_CURR", how="left")

# Add portfolio-level market regime constants from Excel file
for k, v in market_regime.items():
    final_df[k] = v

# Transition target: not 90+ at observation, but reaches 90+ in future window
final_df["TARGET_TRANSITION_TO_90"] = (
    (final_df["OBS_STAGE"].fillna(0) < 3)
    & (final_df["FUTURE_DEFAULT_90_FLAG"].fillna(0) == 1)
).astype("int8")

# Fill missing binary targets with 0 when no future rows are present in horizon
for col in ["FUTURE_DPD30_FLAG", "FUTURE_DPD60_FLAG", "FUTURE_DEFAULT_90_FLAG"]:
    final_df[col] = final_df[col].fillna(0).astype("int8")

print("final_df:", final_df.shape)
print("new columns count:", len(final_df.columns))
final_df.head()


In [ ]:
# Quick target diagnostics on train rows only
train_view = final_df[final_df["is_train"] == 1].copy()

for col in ["FUTURE_DPD30_FLAG", "FUTURE_DPD60_FLAG", "FUTURE_DEFAULT_90_FLAG", "TARGET_TRANSITION_TO_90"]:
    rate = train_view[col].mean()
    print(f"{col:26} -> {rate:.4%}")

print("\nTARGET vs FUTURE_DEFAULT_90_FLAG (row-wise):")
print(pd.crosstab(train_view["TARGET"], train_view["FUTURE_DEFAULT_90_FLAG"], normalize="index"))


In [ ]:
final_df['TARGET_TRANSITION_TO_90'].value_counts()

In [ ]:
# Save
out_csv = OUT_DIR / "home_credit_future_dpd_labels.csv"
out_parquet = OUT_DIR / "home_credit_future_dpd_labels.parquet"

final_df.to_csv(out_csv, index=False)
saved = [str(out_csv)]

try:
    final_df.to_parquet(out_parquet, index=False)
    saved.append(str(out_parquet))
except Exception as e:
    print("Parquet save skipped:", e)

print("Saved:")
for f in saved:
    print(" -", f)


In [ ]:
"obs_reference_month" in final_df.columns 

## 6) Notes / How to Use

- For model training, use only rows with `is_train=1`.
- Recommended default target: `FUTURE_DEFAULT_90_FLAG`.
- Strict no-leakage rule:
  - Features: only `past_*`, `dpd_at_obs`, `OBS_STAGE`, and other columns known at observation time.
  - Do **not** use `future_*` columns as predictors.
- You can tune timeline:
  - `OBS_MONTH`, `PAST_WINDOW`, `FUTURE_HORIZON`
- You can repeat this process for multiple observation points and stack snapshots for larger training data.


## 7) Train EWS Model + Explainability (SHAP + Human Rules)

This section trains a predictive model for future deterioration (`TARGET_TRANSITION_TO_90`) using only observation/past features, then adds explainability:
- global feature importance
- local customer-level SHAP explanation
- compact human-readable decision rules


In [ ]:
pip install shap

In [ ]:
# Model + explainability dependencies
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, confusion_matrix

try:
    import shap
except Exception as e:
    raise ImportError(
        "SHAP is required for explainability. Install with: pip install shap"
    ) from e

import joblib


In [ ]:
# -----------------------------
# Training configuration
# -----------------------------
TARGET_COL = 'TARGET_TRANSITION_TO_90'
POSITIVE_VALUE = 1
TEST_SIZE = 0.20
RANDOM_STATE = 42

# Features allowed for training (strictly past/observation-time information)
LEAK_COLS = {
    'SK_ID_CURR',
    'TARGET',                 # original competition target
    'is_train',
    'future_obs_months',
    'future_dpd_max',
    'FUTURE_DPD30_FLAG',
    'FUTURE_DPD60_FLAG',
    'FUTURE_DEFAULT_90_FLAG',
    'FUTURE_STAGE_MAX',
    'TARGET_TRANSITION_TO_90',
}

train_df = final_df[final_df['is_train'] == 1].copy()
train_df = train_df[train_df[TARGET_COL].notna()].copy()

y = train_df[TARGET_COL].astype('int8')

# Keep numeric predictive features only
candidate_cols = [c for c in train_df.columns if c not in LEAK_COLS]
X_num = train_df[candidate_cols].apply(pd.to_numeric, errors='coerce')

# Keep columns with enough data
valid_cols = [c for c in X_num.columns if X_num[c].notna().mean() >= 0.60]
X = X_num[valid_cols].copy()

# Basic imputation
medians = X.median(numeric_only=True)
X = X.fillna(medians)

print('Train rows:', len(X))
print('Feature count:', X.shape[1])
print('Positive rate:', f"{y.mean():.4%}")

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print('X_train:', X_train.shape, '| X_valid:', X_valid.shape)


In [ ]:
# -----------------------------
# Train main EWS model
# -----------------------------
rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=14,
    min_samples_leaf=40,
    class_weight='balanced_subsample',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

valid_proba = rf.predict_proba(X_valid)[:, 1]
valid_pred = (valid_proba >= 0.50).astype('int8')

auc = roc_auc_score(y_valid, valid_proba)
pr_auc = average_precision_score(y_valid, valid_proba)

print(f"Validation ROC-AUC: {auc:.4f}")
print(f"Validation PR-AUC : {pr_auc:.4f}")
print('Confusion Matrix @0.50:')
print(confusion_matrix(y_valid, valid_pred))
print('Classification Report @0.50:')
print(classification_report(y_valid, valid_pred, digits=4))


In [ ]:
# -----------------------------
# Global importance
# -----------------------------
feat_imp = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf.feature_importances_,
}).sort_values('importance', ascending=False).reset_index(drop=True)

print('Top 20 global important features:')
display(feat_imp.head(20))


In [ ]:
feat_imp

In [ ]:
# -----------------------------
# Human-readable rule extraction (compact tree)
# -----------------------------
rule_tree = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=200,
    class_weight='balanced',
    random_state=RANDOM_STATE,
)
rule_tree.fit(X_train, y_train)


def extract_positive_rules(tree_model, feature_names, base_rate, max_rules=12):
    tree_ = tree_model.tree_
    rules = []

    def walk(node_id, conds):
        left = tree_.children_left[node_id]
        right = tree_.children_right[node_id]

        if left == right:
            counts = tree_.value[node_id][0]
            neg_w = float(counts[0]) if len(counts) >= 1 else 0.0
            pos_w = float(counts[1]) if len(counts) >= 2 else 0.0
            n = int(tree_.n_node_samples[node_id])
            if n <= 0:
                return
            total_w = neg_w + pos_w
            rate = (pos_w / total_w) if total_w > 0 else 0.0
            pred = int(rate >= 0.5)
            if pred == 1:
                lift = (rate / base_rate) if base_rate > 0 else 0.0
                rules.append({
                    'where_clause': ' AND '.join(conds) if conds else '1=1',
                    'sample_n': n,
                    'positive_rate': float(rate),
                    'lift_vs_base': float(lift),
                })
            return

        f_idx = int(tree_.feature[node_id])
        thr = float(tree_.threshold[node_id])
        feat = feature_names[f_idx]

        walk(left, conds + [f'"{feat}" <= {thr:.6g}'])
        walk(right, conds + [f'"{feat}" > {thr:.6g}'])

    walk(0, [])
    out = sorted(rules, key=lambda d: (d['positive_rate'], d['lift_vs_base'], d['sample_n']), reverse=True)
    out = out[:max_rules]
    for i, r in enumerate(out, 1):
        r['rule_name'] = f'EWS_rule_{i}'
    return pd.DataFrame(out)

base_rate = float(y_train.mean())
rule_df = extract_positive_rules(rule_tree, list(X_train.columns), base_rate=base_rate, max_rules=12)

print('Extracted high-risk rules:')
display(rule_df)


In [ ]:
print(len(rule_df))

In [ ]:
# -----------------------------
# SHAP explainability (global + local)
# -----------------------------
explainer = shap.TreeExplainer(rf)

# Use a smaller sample for speed in notebooks
shap_sample_n = min(3000, len(X_valid))
X_shap = X_valid.sample(shap_sample_n, random_state=RANDOM_STATE)

shap_values_raw = explainer.shap_values(X_shap)
if isinstance(shap_values_raw, list):
    shap_values = shap_values_raw[1] if len(shap_values_raw) > 1 else shap_values_raw[0]
else:
    shap_values = shap_values_raw

# Global bar plot
shap.summary_plot(shap_values, X_shap, plot_type='bar', max_display=20)

# Global beeswarm plot
shap.summary_plot(shap_values, X_shap, max_display=20)


In [ ]:
# Local explanation for one validation customer
one_idx = X_valid.index[10]
one_cust = int(train_df.loc[one_idx, 'SK_ID_CURR'])
one_x = X_valid.loc[[one_idx]]
one_prob = float(rf.predict_proba(one_x)[0, 1])

one_sv_raw = explainer.shap_values(one_x)
if isinstance(one_sv_raw, list):
    one_sv = one_sv_raw[1][0] if len(one_sv_raw) > 1 else one_sv_raw[0][0]
else:
    arr = np.asarray(one_sv_raw)
    if arr.ndim == 3:
        one_sv = arr[0, :, 1] if arr.shape[2] > 1 else arr[0, :, 0]
    elif arr.ndim == 2:
        one_sv = arr[0]
    else:
        one_sv = arr

local_exp = pd.DataFrame({
    'feature': one_x.columns,
    'feature_value': one_x.iloc[0].values,
    'shap_value': one_sv,
    'abs_shap': np.abs(one_sv),
}).sort_values('abs_shap', ascending=False)

print('Customer ID:', one_cust)
print('Predicted future 90+ transition probability:', f"{one_prob:.4%}")
print('Top local drivers:')
display(local_exp.head(12))


In [ ]:
one_idx

In [ ]:
PROJECT_DIR

In [ ]:
# -----------------------------
# Save artifacts for API/UI reuse
# -----------------------------
MODEL_DIR = PROJECT_DIR / 'nl_sql_qa' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

artifact = {
    'model_type': 'rf_future_dpd_ews',
    'target_col': TARGET_COL,
    'positive_value': POSITIVE_VALUE,
    'feature_cols': list(X_train.columns),
    'medians': medians.to_dict(),
    'model': rf,
    'rule_tree': rule_tree,
    'top_features': feat_imp.head(50).to_dict(orient='records'),
    'rules': rule_df.to_dict(orient='records'),
    'validation_metrics': {
        'roc_auc': float(auc),
        'pr_auc': float(pr_auc),
        'train_rows': int(len(X_train)),
        'valid_rows': int(len(X_valid)),
        'positive_rate_train': float(y_train.mean()),
    },
}

artifact_path = MODEL_DIR / 'ews_future_dpd_rf_artifact.pkl'
joblib.dump(artifact, artifact_path)

rule_csv_path = OUT_DIR / 'ews_future_dpd_rules.csv'
rule_df.to_csv(rule_csv_path, index=False)

feat_csv_path = OUT_DIR / 'ews_future_dpd_feature_importance.csv'
feat_imp.to_csv(feat_csv_path, index=False)

print('Saved model artifact:', artifact_path)
print('Saved rules CSV    :', rule_csv_path)
print('Saved feature CSV  :', feat_csv_path)
